# Oil Contamination Detection from Satellite Imagery

This notebook demonstrates the machine learning pipeline for detecting oil contamination
impacts on vegetation using Sentinel-2 satellite imagery.

## Reference
- Manuscript Section 2: Materials and Methods
- Supplementary Material S1: Technical Details

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Import project modules
from models.vegetation_indices import calculate_arvi, calculate_savi, calculate_hci, calculate_all_indices
from models.cnn_encoder_decoder import build_classification_model
from utils.visualization import plot_vegetation_indices_timeseries, plot_model_comparison
from utils.metrics import calculate_all_metrics

## 1. Vegetation Index Calculation

Calculate ARVI, SAVI, and HCI indices from multispectral bands.

### ARVI (Equation 1)
$$ARVI = \frac{NIR - RB}{NIR + RB}$$

where $RB = Red - \gamma(Blue - Red)$ and $\gamma = 1.0$

### SAVI (Equation 2)
$$SAVI = \frac{(NIR - Red)}{(NIR + Red + L)} \times (1 + L)$$

where $L = 0.5$ for intermediate vegetation density

### HCI (Equation 3)
$$HCI = \frac{\rho_{2100} - \rho_{660}}{\rho_{2100} + \rho_{660}}$$

In [ ]:
# Example: Calculate vegetation indices from synthetic data
np.random.seed(42)

# Simulate Sentinel-2 band reflectance values
height, width = 256, 256
blue = np.random.uniform(0.02, 0.08, (height, width))   # B02
green = np.random.uniform(0.03, 0.10, (height, width))  # B03
red = np.random.uniform(0.02, 0.12, (height, width))    # B04
nir = np.random.uniform(0.15, 0.50, (height, width))    # B08
swir1 = np.random.uniform(0.10, 0.35, (height, width))  # B11
swir2 = np.random.uniform(0.05, 0.25, (height, width))  # B12

# Calculate indices
arvi = calculate_arvi(nir, red, blue)
savi = calculate_savi(nir, red)
hci = calculate_hci(swir2, red)  # Using SWIR2 as proxy for 2100nm

print(f"ARVI range: [{arvi.min():.3f}, {arvi.max():.3f}]")
print(f"SAVI range: [{savi.min():.3f}, {savi.max():.3f}]")
print(f"HCI range: [{hci.min():.3f}, {hci.max():.3f}]")

In [ ]:
# Visualize vegetation indices
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

im0 = axes[0].imshow(arvi, cmap='RdYlGn', vmin=-1, vmax=1)
axes[0].set_title('ARVI', fontsize=14)
plt.colorbar(im0, ax=axes[0])

im1 = axes[1].imshow(savi, cmap='RdYlGn', vmin=-1, vmax=1)
axes[1].set_title('SAVI', fontsize=14)
plt.colorbar(im1, ax=axes[1])

im2 = axes[2].imshow(hci, cmap='RdYlBu', vmin=-1, vmax=1)
axes[2].set_title('HCI', fontsize=14)
plt.colorbar(im2, ax=axes[2])

plt.tight_layout()
plt.show()

## 2. CNN Encoder-Decoder Model

Build and examine the CNN architecture as described in Section 2.3.

Architecture:
- Encoder: 5 blocks with 32→64→128→256→512 filters
- Decoder: Transposed convolutions with skip connections
- Output: Pixel-wise classification

In [ ]:
# Build the model
input_shape = (256, 256, 6)  # 6 Sentinel-2 bands
n_classes = 4  # Background, Contaminated, Recovering, Recovered

model = build_classification_model(
    input_shape=input_shape,
    n_classes=n_classes,
    filters=[32, 64, 128, 256, 512],
    dropout_rate=0.3
)

print(f"Total parameters: {model.count_params():,}")
model.summary()

## 3. Model Performance Comparison

Compare performance metrics from Supplementary Material S1.8.

In [ ]:
# Benchmark results from the manuscript
models = ['SVM', 'RF', 'XGBoost', 'VGG-16', 'ResNet-50', 'U-Net', 'DeepLabV3+', 'CNN (Ours)']

metrics = {
    'Accuracy': [0.793, 0.820, 0.835, 0.817, 0.841, 0.872, 0.864, 0.893],
    'F1 Score': [0.76, 0.79, 0.81, 0.79, 0.82, 0.85, 0.84, 0.88],
    "Cohen's κ": [0.58, 0.62, 0.65, 0.61, 0.67, 0.73, 0.71, 0.76]
}

# Create comparison plot
plot_model_comparison(models, metrics, figsize=(14, 6))

## 4. Temporal Analysis Example

Demonstrate FFT analysis for detecting periodic patterns in vegetation recovery.

In [ ]:
from analysis.temporal_analysis import fft_analysis

# Simulate vegetation index time series (monthly data over 10 years)
n_months = 120
t = np.arange(n_months)

# Create synthetic signal with annual cycle and trend
annual_cycle = 0.1 * np.sin(2 * np.pi * t / 12)  # Annual seasonality
trend = 0.002 * t  # Gradual recovery
noise = np.random.normal(0, 0.02, n_months)
vegetation_signal = 0.3 + annual_cycle + trend + noise

# Perform FFT analysis
fft_results = fft_analysis(
    vegetation_signal,
    sampling_rate=12,  # 12 samples per year (monthly)
    freq_range=(0.1, 6)  # 0.1 to 6 cycles per year
)

print(f"Dominant frequency: {fft_results['dominant_frequency']:.2f} cycles/year")
print(f"Significant peaks: {fft_results['significant_peaks']}")

In [ ]:
# Plot time series and FFT spectrum
fig, axes = plt.subplots(2, 1, figsize=(12, 8))

# Time series
axes[0].plot(t, vegetation_signal, 'b-', linewidth=1)
axes[0].set_xlabel('Month')
axes[0].set_ylabel('Vegetation Index')
axes[0].set_title('Vegetation Index Time Series')
axes[0].grid(True, alpha=0.3)

# FFT spectrum
axes[1].plot(fft_results['frequencies'], fft_results['power'], 'b-', linewidth=1.5)
axes[1].axvline(x=1.0, color='r', linestyle='--', label='Annual (1 cycle/yr)')
axes[1].set_xlabel('Frequency (cycles/year)')
axes[1].set_ylabel('Power')
axes[1].set_title('FFT Power Spectrum')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Spatial Analysis Example

Calculate fractal dimension using box-counting method.

In [ ]:
from analysis.spatial_analysis import fractal_dimension_box_counting

# Create synthetic binary contamination mask
np.random.seed(42)
mask = np.random.random((512, 512)) > 0.7  # ~30% contaminated

# Apply some spatial structure
from scipy.ndimage import gaussian_filter
mask_smooth = gaussian_filter(mask.astype(float), sigma=5) > 0.3

# Calculate fractal dimension
fractal_results = fractal_dimension_box_counting(
    mask_smooth,
    pixel_size=30  # 30m Sentinel-2 resolution
)

print(f"Fractal Dimension: {fractal_results['fractal_dimension']:.3f}")
print(f"R² of fit: {fractal_results['r_squared']:.3f}")

In [ ]:
# Visualize the mask and box-counting relationship
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Contamination mask
axes[0].imshow(mask_smooth, cmap='Reds')
axes[0].set_title('Contamination Mask', fontsize=14)
axes[0].axis('off')

# Log-log plot for fractal dimension
axes[1].loglog(fractal_results['box_sizes'], fractal_results['box_counts'], 'bo-', markersize=8)
axes[1].set_xlabel('Box Size (m)', fontsize=12)
axes[1].set_ylabel('Box Count', fontsize=12)
axes[1].set_title(f"Box-Counting (D = {fractal_results['fractal_dimension']:.3f})", fontsize=14)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Summary

This notebook demonstrated:

1. **Vegetation Index Calculation**: ARVI, SAVI, and HCI indices for assessing vegetation health and hydrocarbon contamination
2. **CNN Architecture**: Encoder-decoder with skip connections achieving 89.3% accuracy
3. **Temporal Analysis**: FFT for detecting periodic recovery patterns
4. **Spatial Analysis**: Fractal dimension for characterizing contamination patterns

For full implementation details, see:
- `models/` - CNN and vegetation index implementations
- `analysis/` - Temporal and spatial analysis functions
- `benchmarks/` - Comparison model implementations